In [2]:
import pandas as pd
import sqlite3
import os

df = pd.read_excel("/run/media/bharathy/DATA/Sales_Dataset_2024.xlsx")
df.columns = df.columns.str.replace(" ", "_").str.replace("-", "_")

os.makedirs("../data", exist_ok=True)
conn = sqlite3.connect("../data/superstore.db")
df.to_sql("orders", conn, if_exists="replace", index=False)
conn.close()

print("Loaded into superstore.db as table 'orders'")
print(df.shape)
print(df.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: '/run/media/bharathy/DATA/Sales_Dataset_2024.xlsx'

In [ ]:
from google import genai
from dotenv import load_dotenv
import os
import re

load_dotenv(dotenv_path="../.env")
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

SCHEMA = """
Table: orders
Columns:
- Date (datetime) — order date
- Region (text) — North/South/East/West
- Product (text) — product name e.g. Smartwatch, Monitor, Mobile, Headphones
- Salesperson (text) — name of salesperson
- Units_Sold (float)
- Unit_Price (float)
- Category (text) — Accessories/Office/Electronics
- Revenue (float)
- Cost (float)
- Profit (float)
"""

print("Setup complete")

In [ ]:
def generate_sql(question: str, error_context: str = None) -> str:
    error_note = f"\n\nYour previous attempt failed with this error: {error_context}\nFix the query." if error_context else ""
    
    prompt = f"""You are a SQLite expert. Given this table schema:

{SCHEMA}

IMPORTANT: This is SQLite, not PostgreSQL or MySQL. Use SQLite syntax only.
- For dates, use strftime('%Y', Date), strftime('%m', Date), or strftime('%Y-%m', Date)

Write a single valid SQLite SELECT query to answer this question:
"{question}"

Rules:
- Only output the raw SQL query, nothing else — no markdown, no explanation, no backticks
- Only use SELECT statements — never DROP, DELETE, UPDATE, INSERT, or ALTER
- Use the exact column and table names given above{error_note}
"""
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    raw = response.text.strip()

    # Strip markdown code fences (handles ```sql ... ``` in any position)
    raw = re.sub(r"^```(?:sql)?\s*|\s*```$", "", raw.strip(), flags=re.MULTILINE).strip()

    return raw

print("generate_sql defined")

In [ ]:
def run_query(sql: str) -> pd.DataFrame:
    sql = sql.strip()
    
    if not sql.upper().startswith("SELECT"):
        raise ValueError(f"Only SELECT queries are allowed. Got: {sql[:50]}")
    
    conn = sqlite3.connect("../data/superstore.db")
    result = pd.read_sql(sql, conn)
    conn.close()
    return result

print("run_query defined")

In [ ]:
def ask(question: str, max_retries: int = 2):
    error_context = None
    sql = None
    for attempt in range(max_retries + 1):
        sql = generate_sql(question, error_context)
        try:
            result = run_query(sql)
            return sql, result
        except Exception as e:
            error_context = str(e)
            print(f"Attempt {attempt+1} failed: {error_context}")
    raise RuntimeError(f"Failed after {max_retries+1} attempts. Last SQL:\n{sql}")

print("ask defined")

In [ ]:
##sql, result = ask("Which region had the lowest monthly revenue and which month?")
#print("Final SQL:", sql)
#result

In [ ]:
def explain_result(question: str, sql: str, result: pd.DataFrame) -> str:
    
    result_text = result.to_string(index=False)
    
    prompt = f"""A user asked this question about sales data:
"{question}"

This SQL query was run to answer it:
{sql}

Here is the result:
{result_text}

Write a short, clear, plain-English answer to the user's original question based on this result. 
Be direct and conversational — 1-2 sentences. Don't mention SQL or the query. Just answer the question naturally, like a data analyst explaining a finding to a colleague.
"""
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    return response.text.strip()

print("explain_result defined")

In [ ]:
def ask_and_explain(question: str):
    sql, result = ask(question)
    explanation = explain_result(question, sql, result)
    
    print("Question:", question)
    print("\nSQL:", sql)
    print("\nResult:")
    print(result)
    print("\nAnswer:", explanation)
    
    return sql, result, explanation
#ask_and_explain("Which region had the lowest monthly revenue and which month?")

In [ ]:
#ask_and_explain("Who is the top salesperson by total revenue?")

In [ ]:
#ask_and_explain("What's the total profit by category?")

In [ ]:
#ask_and_explain("How many units were sold in the West region?")

In [ ]:
import pandas as pd

df = pd.read_excel("/run/media/bharathy/DATA/Sales_Dataset_2024.xlsx")
df.columns = df.columns.str.replace(" ", "_").str.replace("-", "_")

# Check unique values in categorical columns for typos/inconsistencies
for col in ["Region", "Product", "Salesperson", "Category"]:
    print(f"\n--- {col} ---")
    print(df[col].unique())

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # Standardize Region: fix case + typos
    region_map = {
        "north": "North", "NORTH": "North",
        "south": "South",
        "east": "East", "easst": "East", "Easst": "East",
        "west": "West", "westt": "West"
    }
    df["Region"] = df["Region"].replace(region_map)
    df["Region"] = df["Region"].str.strip().str.title()
    
    # Standardize Product: fix case + typos
    product_map = {
        "moblie": "Mobile", "MOBLIE": "Mobile",
        "smartwatch": "Smartwatch", "SMARTWATCH": "Smartwatch",
        "headphones": "Headphones", "headPhones": "Headphones",
        "laptop": "Laptop",
        "tabllet": "Tablet"
    }
    df["Product"] = df["Product"].replace(product_map)
    df["Product"] = df["Product"].str.strip().str.title()
    
    # Report nulls before dropping/handling
    print("Nulls before cleaning:")
    print(df[["Region", "Product", "Salesperson"]].isnull().sum())
    
    return df

df_clean = clean_data(df)

print("\n--- Region after cleaning ---")
print(df_clean["Region"].unique())
print("\n--- Product after cleaning ---")
print(df_clean["Product"].unique())

In [ ]:
#removin null values that makes up 2% try1

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    original_count = len(df)
    
    region_map = {
        "north": "North", "NORTH": "North",
        "south": "South",
        "east": "East", "easst": "East", "Easst": "East",
        "west": "West", "westt": "West"
    }
    df["Region"] = df["Region"].replace(region_map)
    df["Region"] = df["Region"].str.strip().str.title()
    
   
    product_map = {
        "moblie": "Mobile", "MOBLIE": "Mobile",
        "smartwatch": "Smartwatch", "SMARTWATCH": "Smartwatch",
        "headphones": "Headphones", "headPhones": "Headphones",
        "laptop": "Laptop",
        "tabllet": "Tablet"
    }
    df["Product"] = df["Product"].replace(product_map)
    df["Product"] = df["Product"].str.strip().str.title()
    
    # Drop rows with nulls in key columns
    df = df.dropna(subset=["Region", "Product", "Salesperson"])
    dropped_count = original_count - len(df)
    
    print(f"Cleaned data: dropped {dropped_count} rows with nulls ({dropped_count/original_count*100:.1f}% of data)")
    print(f"Remaining rows: {len(df)}")
    
    return df

df_clean = clean_data(df)

In [ ]:
import sqlite3
conn = sqlite3.connect("../data/superstore.db")
df_clean.to_sql("orders", conn, if_exists="replace", index=False)
conn.close()
print("Cleaned data saved to superstore.db")

In [ ]:
ask_and_explain("Which region had the lowest monthly revenue and which month?")